# Simulación secuencial — lote 2 de 4

Dos GPU, una configuración en cada una, 100 copias por configuración, radio 18,5 y cinco capas. Enfriamiento secuencial de 20,0 a 6,0 K con paso 0,1 K; 25.000 barridos de termalización y 10.000 de medición en cada temperatura. Cada estado comienza con espines aleatorios; la configuración final de cada temperatura inicia la siguiente.

Estados: **3** (kDM=0.33, Kan1=0.05, Hex=0) y **4** (kDM=0.6, Kan1=0.05, Hex=0), gamma=90°. Metropolis con tablero de ajedrez y direcciones generadas por transformada inversa, de acuerdo con el código corregido.

Se guarda un archivo por temperatura y estado: vectores de espín de las 100 copias y las seis series de cada copia, una medición por barrido. La máscara permite reconstruir la malla. Los archivos se validan antes de declararlos completos; los resultados, tiempos y registros de memoria se encuentran en `/kaggle/working/lote_02`. Los valores de energía están normalizados por Eo, que se guarda. El cálculo de curvas de calor específico y susceptibilidad y su incertidumbre corresponde al cuaderno de análisis.

Este procedimiento registra resultados, pero no garantiza por sí solo la convergencia. Los checkpoints registran las claves y configuraciones finales; este cuaderno todavía no implementa la reanudación automática.

In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
BLOQUE = 1000
import jax
import jax.numpy as jnp
from jax import random, pmap, jit, lax
from jax.scipy.special import gammaincc
from functools import partial
import numpy as np
import time
import math
import os

# ==========================================================================
# BLOQUES DEL CURSO CUBIERTOS POR ESTE CODIGO
# --------------------------------------------------------------------------
# [B4] MCMC: Metropolis de un solo espin con propuesta simetrica y actualizacion
#      en tablero de ajedrez. NO se modifica (mismas formulas, misma energia,
#      misma normalizacion por Eo).
# [B1] Generacion de variables aleatorias: la direccion de propuesta y el estado
#      inicial se generan por TRANSFORMADA INVERSA a partir de U(0,1):
#      cos(theta)=2u-1, phi=2*pi*v. Prueba de uniformidad chi-cuadrado opcional.
# [B2] Monte Carlo + remuestreo: jackknife por bloques, tiempo de autocorrelacion
#      y bootstrap entre replicas para Cv, Chi, E, M y temperaturas de pico.
#      Se activa/desactiva con la bandera CALCULAR_ERRORES.
#
# Con CALCULAR_ERRORES = False el flujo es el de siempre (solo Datos_*.npz).
# Con CALCULAR_ERRORES = True se genera ademas Stats_*.npz por estado.
# ==========================================================================

# Verificación de entorno Multi-GPU
num_gpus = jax.local_device_count(backend="gpu")
assert num_gpus == 2, f"Esta prueba requiere exactamente 2 GPU, detectadas: {num_gpus}"
jax.config.update("jax_enable_x64", False) # Mantener FP32 para velocidad

# ==========================================
# BANDERAS DE EJECUCION
# ==========================================
CALCULAR_ERRORES = False       # True: jackknife + tau + bootstrap (Stats_*.npz). False: flujo original.
PRUEBA_UNIFORMIDAD = False      # True: chi-cuadrado de uniformidad del generador de direcciones (B1)
METODO_ESFERA = "inversa"      # "inversa" (transformada inversa) | "gauss" (normalizar gaussianas)

# ==========================================
# PARÁMETROS FIJOS (GLOBALES)
# ==========================================
NUM_ESTADOS = 2
REPLICAS_POR_ESTADO = 100
MCS_TERM = 25000
MCS_MEAS = 10000
MCS_MAX = MCS_TERM + MCS_MEAS

Rd = 18.5 
L = 5
So = 1.0
Ti = 20.0
Tf = 6.0
dT = -0.1
kB = 0.086173404
L2 = int(Rd) + 1
Ntot_grid = (2*L2 + 1) * (2*L2 + 1) * (L + 2) 

# ==========================================
# PARÁMETROS DE LA ESTADISTICA DE ERRORES (solo se usan si CALCULAR_ERRORES)
# ==========================================
N_BLOQUES_JK = 10             # bloques del jackknife (MCS_MEAS debe ser divisible)
SOKAL_C = 5.0                 # constante de la ventana automatica de Sokal
NBOOT = 1000                  # remuestras bootstrap entre replicas
UNIF_N_POR_GPU = 2_000_000    # muestras por GPU en la prueba de uniformidad
UNIF_BINS = 64

NOMBRES_SERIES = ("E", "M", "E_exc", "E_an", "E_dm", "E_hex")
NOMBRES_BOOT = NOMBRES_SERIES + ("Cv", "Chi")
assert MCS_MEAS % N_BLOQUES_JK == 0, "MCS_MEAS debe ser multiplo de N_BLOQUES_JK"

# # Genera una semilla basada en hardware/OS
# np_seed = int.from_bytes(os.urandom(4), 'little')
# np.random.seed(np_seed)
# # Generación de parámetros (se mantiene tu código, pero ahora será distinto cada vez)
# Jex_base  = [1.0 for _ in range(NUM_ESTADOS)]
# kDM_base  = [round(np.random.uniform(0.0, 1.5), 2) for _ in range(NUM_ESTADOS)]
# Kan1_base = [round(np.random.uniform(0.0, 0.6), 2) for _ in range(NUM_ESTADOS)]
# Hex_base  = [round(np.random.uniform(0.0, 1.5), 2) for _ in range(NUM_ESTADOS)]
# gamma_base= [90.0 for _ in range(NUM_ESTADOS)] 
# gamma_rad_base = [math.radians(g) for g in gamma_base]

NUM_ESTADOS = 2

# Valores fijos para los dos estados de control
Jex_base  = [1.0, 1.0]
kDM_base  = [0.33, 0.6]
Kan1_base = [0.05, 0.05]
Hex_base  = [0, 0]
gamma_base= [90, 90] 
gamma_rad_base = [math.radians(g) for g in gamma_base]

assert NUM_ESTADOS % num_gpus == 0, "NUM_ESTADOS debe ser multiplo del numero de GPUs"

# Preparación de tensores divididos para jax.pmap (Num_GPUs, Replicas_por_GPU, ...)
estados_por_gpu = NUM_ESTADOS // num_gpus
batch_por_gpu = estados_por_gpu * REPLICAS_POR_ESTADO

def format_param(base_list):
    arr = np.array(base_list).repeat(REPLICAS_POR_ESTADO)
    return arr.reshape(num_gpus, batch_por_gpu, 1, 1, 1)

Jex_pmap = format_param(Jex_base)
kDM_pmap = format_param(kDM_base)
Kan1_pmap = format_param(Kan1_base)
Hex_pmap = format_param(Hex_base)
gamma_pmap = format_param(gamma_rad_base)
Eo_pmap = (Jex_pmap * Ntot_grid * 6 * So**2 + kDM_pmap * Ntot_grid * 6 * So**2 + Hex_pmap * Ntot_grid * So).reshape(num_gpus, batch_por_gpu)

# ==========================================
# PRECALCULO DE MÁSCARAS (CPU -> JAX)
# ==========================================
x_coords = np.arange(-L2, L2 + 1).reshape(1, 1, 1, -1)
y_coords = np.arange(-L2, L2 + 1).reshape(1, 1, -1, 1)
z_coords = np.arange(1, L + 1).reshape(1, -1, 1, 1)

radii_sq = x_coords**2 + y_coords**2
disco_mask_2d = (radii_sq <= Rd**2)

# Equivalente exacto de .expand(1, L, 2*L2 + 1, 2*L2 + 1)
disco_mask_3d = np.broadcast_to(disco_mask_2d, (1, L, 2*L2 + 1, 2*L2 + 1))

checkerboard = (x_coords + y_coords + z_coords) % 2

mask_even_np = ((checkerboard == 0) & disco_mask_3d)[..., np.newaxis]
mask_odd_np  = ((checkerboard == 1) & disco_mask_3d)[..., np.newaxis]
disco_mask_np = disco_mask_3d[..., np.newaxis]

N_spins = int(disco_mask_3d.sum()) 

# Transferir a dispositivos JAX
mask_even_jnp = jnp.array(mask_even_np)
mask_odd_jnp = jnp.array(mask_odd_np)
disco_mask_jnp = jnp.array(disco_mask_np)

# ==========================================
# [B1] GENERACION DE DIRECCIONES ALEATORIAS UNIFORMES EN LA ESFERA
# ==========================================
def esfera_uniforme(key, shape, metodo=METODO_ESFERA):
    """Vectores unitarios uniformes en S^2 con forma shape + (3,).
    'inversa': dOmega = sin(theta) dtheta dphi  =>  cos(theta) ~ U[-1,1], phi ~ U[0,2pi).
               Transformada inversa aplicada a dos U(0,1) independientes.
    'gauss'  : normalizar una gaussiana 3D (metodo anterior, se conserva para comparar)."""
    if metodo == "inversa":
        u = random.uniform(key, shape + (2,))
        cos_t = 2.0 * u[..., 0] - 1.0
        phi = 2.0 * jnp.pi * u[..., 1]
        sin_t = jnp.sqrt(jnp.maximum(1.0 - cos_t * cos_t, 0.0))
        return jnp.stack([sin_t * jnp.cos(phi), sin_t * jnp.sin(phi), cos_t], axis=-1)
    elif metodo == "gauss":
        v = random.normal(key, shape + (3,))
        return v / jnp.sqrt(jnp.sum(v * v, axis=-1, keepdims=True))
    raise ValueError(f"metodo de esfera desconocido: {metodo}")

@pmap
def _muestras_esfera_pmap(key):
    """Cada GPU genera su propia muestra (claves distintas) con ambos metodos y devuelve
    histogramas de cos(theta) y phi, y momentos de primer/segundo orden."""
    ks = random.split(key, 2)
    salida = []
    for i, metodo in enumerate(("inversa", "gauss")):
        v = esfera_uniforme(ks[i], (UNIF_N_POR_GPU,), metodo)
        z = v[:, 2]
        phi = jnp.arctan2(v[:, 1], v[:, 0])
        iz = jnp.clip(jnp.floor((z + 1.0) / 2.0 * UNIF_BINS).astype(jnp.int32), 0, UNIF_BINS - 1)
        ip = jnp.clip(jnp.floor((phi + jnp.pi) / (2.0 * jnp.pi) * UNIF_BINS).astype(jnp.int32), 0, UNIF_BINS - 1)
        salida.append((jnp.bincount(iz, length=UNIF_BINS),
                       jnp.bincount(ip, length=UNIF_BINS),
                       jnp.mean(v, axis=0),
                       jnp.mean(v * v, axis=0)))
    return tuple(salida)

def prueba_uniformidad_esfera(key):
    """Chi-cuadrado de bondad de ajuste (cos(theta) y phi uniformes) + momentos. Se ejecuta una vez."""
    res = _muestras_esfera_pmap(random.split(key, num_gpus))
    informe = {}
    for metodo, r in zip(("inversa", "gauss"), res):
        cz = np.asarray(r[0]).sum(axis=0).astype(np.float64)
        cp = np.asarray(r[1]).sum(axis=0).astype(np.float64)
        m1 = np.asarray(r[2]).mean(axis=0)
        m2 = np.asarray(r[3]).mean(axis=0)
        n = cz.sum()
        esperado = n / UNIF_BINS
        chi_z = float(((cz - esperado)**2 / esperado).sum())
        chi_p = float(((cp - esperado)**2 / esperado).sum())
        df = UNIF_BINS - 1
        p_z = float(gammaincc(df / 2.0, chi_z / 2.0))
        p_p = float(gammaincc(df / 2.0, chi_p / 2.0))
        informe[metodo] = dict(chi2_cos=chi_z, p_cos=p_z, chi2_phi=chi_p, p_phi=p_p,
                               media_xyz=m1, segundo_momento_xyz=m2, n=n)
        print(f"[B1] Uniformidad '{metodo}': chi2(cos)={chi_z:.1f} p={p_z:.3f} | chi2(phi)={chi_p:.1f} p={p_p:.3f} "
              f"| <v>={np.round(m1, 4)} | <v^2>={np.round(m2, 4)} (esperado 0 y 0.3333) | N={int(n)}")
    return informe

# ==========================================
# FUNCIONES JAX COMPILADAS  [B4]
# ==========================================
def compute_energy_components(spins, mask_vec, kDM, Jex, Kan1, Hex_mag, gamma_rad, So):
    S_xp = jnp.roll(spins, -1, axis=3); S_xm = jnp.roll(spins, 1, axis=3)
    S_yp = jnp.roll(spins, -1, axis=2); S_ym = jnp.roll(spins, 1, axis=2)
    S_zp = jnp.roll(spins, -1, axis=1); S_zm = jnp.roll(spins, 1, axis=1)
    S_zp = S_zp.at[:, -1, :, :, :].set(0.0)
    S_zm = S_zm.at[:, 0, :, :, :].set(0.0)
    sum_vecinos = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm
    
    E_exc = -Jex * jnp.sum(spins * sum_vecinos, axis=-1) * 0.5
    E_an1 = (Kan1 / So**4) * ((spins[...,0]*spins[...,1])**2 + (spins[...,0]*spins[...,2])**2 + (spins[...,1]*spins[...,2])**2)
    dm_x = spins[...,1]*(S_xp[...,2] - S_xm[...,2]) - spins[...,2]*(S_xp[...,1] - S_xm[...,1])
    dm_y = spins[...,2]*(S_yp[...,0] - S_ym[...,0]) - spins[...,0]*(S_yp[...,2] - S_ym[...,2])
    E_dm = -kDM * (dm_x + dm_y) * 0.5
    E_hex = -Hex_mag * (spins[...,0]*jnp.cos(gamma_rad) + spins[...,2]*jnp.sin(gamma_rad))
    
    mask = jnp.squeeze(mask_vec, axis=-1)
    return (jnp.sum(E_exc * mask, axis=(1,2,3)),
            jnp.sum(E_an1 * mask, axis=(1,2,3)),
            jnp.sum(E_dm * mask, axis=(1,2,3)),
            jnp.sum(E_hex * mask, axis=(1,2,3)))

def metropolis_substep(spins, key, beta, mask_vec, kDM, Jex, Kan1, Hex_mag, gamma_rad):
    S_xp = jnp.roll(spins, -1, axis=3); S_xm = jnp.roll(spins, 1, axis=3)
    S_yp = jnp.roll(spins, -1, axis=2); S_ym = jnp.roll(spins, 1, axis=2)
    S_zp = jnp.roll(spins, -1, axis=1); S_zm = jnp.roll(spins, 1, axis=1)
    S_zp = S_zp.at[:, -1, :, :, :].set(0.0)
    S_zm = S_zm.at[:, 0, :, :, :].set(0.0)
    
    key_t, key_a = random.split(key)
    # [B1] propuesta: direccion uniforme en la esfera por transformada inversa (antes: normal + normalizar)
    spins_trial = esfera_uniforme(key_t, spins.shape[:-1]) * So
    spins_trial = jnp.where(mask_vec, spins_trial, spins)
    dS = spins_trial - spins
    
    sum_vecinos = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm
    dE_exc = -Jex * jnp.sum(dS * sum_vecinos, axis=-1)
    E_an1_t = (Kan1 / So**4) * ((spins_trial[...,0]*spins_trial[...,1])**2 + (spins_trial[...,0]*spins_trial[...,2])**2 + (spins_trial[...,1]*spins_trial[...,2])**2)
    E_an1_s = (Kan1 / So**4) * ((spins[...,0]*spins[...,1])**2 + (spins[...,0]*spins[...,2])**2 + (spins[...,1]*spins[...,2])**2)
    dE_an1 = E_an1_t - E_an1_s
    d_dm_x = dS[...,1]*(S_xp[...,2] - S_xm[...,2]) - dS[...,2]*(S_xp[...,1] - S_xm[...,1])
    d_dm_y = dS[...,2]*(S_yp[...,0] - S_ym[...,0]) - dS[...,0]*(S_yp[...,2] - S_ym[...,2])
    dE_dm = -kDM * (d_dm_x + d_dm_y)
    dE_hex = -Hex_mag * (dS[...,0]*jnp.cos(gamma_rad) + dS[...,2]*jnp.sin(gamma_rad))
    delta_E = dE_exc + dE_an1 + dE_dm + dE_hex
    
    rand_tensor = random.uniform(key_a, delta_E.shape)
    accept = (delta_E < 0) | (rand_tensor < jnp.exp(-delta_E * beta))
    accept_mask = jnp.expand_dims(accept, -1) & mask_vec
    
    return jnp.where(accept_mask, spins_trial, spins)

# Bucle compilado en GPU de termalización pura
def term_loop(spins, key, beta, mask_even, mask_odd, kDM, Jex, Kan1, Hex_mag, gamma_rad):
    def body_fn(i, val):
        s, k = val
        k, k1, k2 = random.split(k, 3)
        s = metropolis_substep(s, k1, beta, mask_even, kDM, Jex, Kan1, Hex_mag, gamma_rad)
        s = metropolis_substep(s, k2, beta, mask_odd, kDM, Jex, Kan1, Hex_mag, gamma_rad)
        return s, k
    return jax.lax.fori_loop(0, BLOQUE, body_fn, (spins, key))

# Bucle compilado en GPU que acumula mediciones paso a paso sin escribir en disco
def meas_loop(spins, key, beta, mask_even, mask_odd, mask_disco, kDM, Jex, Kan1, Hex_mag, gamma_rad, Eo_batch):
    def step_fn(val, _):
        s, k = val
        k, k1, k2 = random.split(k, 3)
        s = metropolis_substep(s, k1, beta, mask_even, kDM, Jex, Kan1, Hex_mag, gamma_rad)
        s = metropolis_substep(s, k2, beta, mask_odd, kDM, Jex, Kan1, Hex_mag, gamma_rad)
        
        e_jex, e_kan, e_kdm, e_hex = compute_energy_components(s, mask_disco, kDM, Jex, Kan1, Hex_mag, gamma_rad, So)
        Ene_batch = e_jex + e_kan + e_kdm + e_hex 
        M_vec = jnp.sum(s * mask_disco, axis=(1,2,3))
        M_norm = jnp.sqrt(jnp.sum(M_vec * M_vec, axis=-1)) / N_spins   # misma norma euclidea, sin linalg.norm
        
        # Guardar en el historial (gestionado internamente por lax.scan)
        metrics = (Ene_batch / Eo_batch, M_norm, e_jex / Eo_batch, e_kan / Eo_batch, e_kdm / Eo_batch, e_hex / Eo_batch)
        return (s, k), metrics
    return jax.lax.scan(step_fn, (spins, key), None, length=BLOQUE)



In [ ]:
# El núcleo term_loop y meas_loop es el original: aquí se ejecuta en bloques
# de 1.000 barridos para medir por separado cada etapa y mostrar progreso.
@partial(pmap, in_axes=(0,0,None,None,None,0,0,0,0,0))
def term_bloque(spins,key,beta,mask_even,mask_odd,kDM,Jex,Kan1,Hex_mag,gamma_rad):
    return term_loop(spins,key,beta,mask_even,mask_odd,kDM,Jex,Kan1,Hex_mag,gamma_rad)

@partial(pmap, in_axes=(0,0,None,None,None,None,0,0,0,0,0,0))
def medir_bloque(spins,key,beta,mask_even,mask_odd,mask_disco,kDM,Jex,Kan1,Hex_mag,gamma_rad,Eo_batch):
    return meas_loop(spins,key,beta,mask_even,mask_odd,mask_disco,kDM,Jex,Kan1,Hex_mag,gamma_rad,Eo_batch)


In [ ]:
import json, threading, subprocess
from pathlib import Path
import psutil
OUT=Path('/kaggle/working/lote_02'); OUT.mkdir(parents=True,exist_ok=True)
proceso=psutil.Process(); monitor_datos=[]; parar=threading.Event(); contexto={'etapa':'inicio','temperatura':None,'barridos':0}
def monitor():
 while not parar.is_set():
    try:
        filas=subprocess.check_output(['nvidia-smi','--query-gpu=index,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],text=True,timeout=3).strip().splitlines()
        gpu=[dict(indice=int(a[0]),usada_mib=float(a[1]),total_mib=float(a[2]),utilizacion=float(a[3])) for a in (x.split(',') for x in filas)]
    except Exception: gpu=[]
    monitor_datos.append(dict(epoch=time.time(),rss_mib=proceso.memory_info().rss/2**20,gpu=gpu,**contexto.copy()))
    parar.wait(1)
thread=threading.Thread(target=monitor,daemon=True); thread.start()


In [ ]:
tiempos=[]; inicio_total=time.perf_counter()
print(f'INICIO | GPU={num_gpus} | estados={NUM_ESTADOS} | copias={REPLICAS_POR_ESTADO} | espines={N_spins}',flush=True)
print('Un estado en cada GPU; 25.000 termalización + 10.000 medición por temperatura.',flush=True)
try:
    inicio=time.perf_counter(); key=random.PRNGKey(43); key_init,key_run=random.split(key); keys=random.split(key_run,num_gpus)
    spins=esfera_uniforme(key_init,(num_gpus,batch_por_gpu,L,2*L2+1,2*L2+1))*So
    spins=spins*disco_mask_jnp; spins.block_until_ready()
    tiempos.append(dict(etapa='inicializacion',segundos=time.perf_counter()-inicio))
    for t10 in range(round(Ti * 10), round(Tf * 10) - 1, -1):
        T=t10/10; beta=1/(kB*T); contexto.update(temperatura=T,etapa='termalizacion',barridos=0)
        inicio=time.perf_counter()
        for i in range(MCS_TERM//BLOQUE):
            spins,keys=term_bloque(spins,keys,beta,mask_even_jnp,mask_odd_jnp,kDM_pmap,Jex_pmap,Kan1_pmap,Hex_pmap,gamma_pmap); spins.block_until_ready()
            contexto['barridos']=(i+1)*BLOQUE
        tiempos.append(dict(temperatura=T,etapa='termalizacion',segundos=time.perf_counter()-inicio))
        contexto.update(etapa='medicion',barridos=0); inicio=time.perf_counter(); series=[[] for _ in range(6)]
        for i in range(MCS_MEAS//BLOQUE):
            (spins,keys),metricas=medir_bloque(spins,keys,beta,mask_even_jnp,mask_odd_jnp,disco_mask_jnp,kDM_pmap,Jex_pmap,Kan1_pmap,Hex_pmap,gamma_pmap,Eo_pmap); spins.block_until_ready()
            for acumulador,valor in zip(series,metricas): acumulador.append(np.asarray(valor))
            contexto['barridos']=(i+1)*BLOQUE
        tiempos.append(dict(temperatura=T,etapa='medicion',segundos=time.perf_counter()-inicio))
        contexto['etapa']='guardado'; inicio=time.perf_counter()
        spins_cpu=np.asarray(spins).reshape(NUM_ESTADOS,REPLICAS_POR_ESTADO,L,2*L2+1,2*L2+1,3)
        series_cpu=[np.concatenate(x,axis=1).reshape(NUM_ESTADOS,MCS_MEAS,REPLICAS_POR_ESTADO) for x in series]
        for e in range(NUM_ESTADOS):
            directorio=OUT/f'estado_{e+3}'; directorio.mkdir(exist_ok=True)
            archivo=directorio/f'T_{T:.1f}.npz'
            temporal=archivo.with_suffix('.tmp')
            vectores=spins_cpu[e][:,disco_mask_np[0,...,0]]
            assert vectores.shape == (REPLICAS_POR_ESTADO,N_spins,3)
            assert np.isfinite(vectores).all() and np.max(np.abs(np.linalg.norm(vectores,axis=-1)-So)) < 1e-3
            with temporal.open('wb') as f:
                np.savez_compressed(f,spins=vectores,mascara=disco_mask_np[0,...,0],
                    **dict(zip(NOMBRES_SERIES,[a[e] for a in series_cpu])),T=T,Jex=Jex_base[e],kDM=kDM_base[e],Kan1=Kan1_base[e],Hex=Hex_base[e],gamma=gamma_base[e],Rd=Rd,So=So,kB=kB,Eo=float(np.asarray(Eo_pmap).reshape(NUM_ESTADOS,REPLICAS_POR_ESTADO)[e,0]),term=MCS_TERM,med=MCS_MEAS,semilla=43,keys=np.asarray(keys),formato='nanodiscos-secuencial-v1')
                f.flush(); os.fsync(f.fileno())
            # Cargar todos los arrays comprueba también el CRC del ZIP y verifica el contenido.
            with np.load(temporal,allow_pickle=False) as ver:
                assert ver['spins'].shape == (REPLICAS_POR_ESTADO,N_spins,3)
                assert ver['mascara'].shape == (L,2*L2+1,2*L2+1) and int(ver['mascara'].sum()) == N_spins
                for nombre in NOMBRES_SERIES:
                    serie=ver[nombre]
                    assert serie.shape == (MCS_MEAS,REPLICAS_POR_ESTADO) and np.isfinite(serie).all(), nombre
                assert abs(float(ver['T'])-T) < 1e-6
                assert np.isfinite(ver['spins']).all()
                for nombre in ('Jex','kDM','Kan1','Hex','gamma','Rd','So','kB','Eo','term','med','semilla','keys','formato'): ver[nombre]
            os.replace(temporal,archivo)
        tiempos.append(dict(temperatura=T,etapa='guardado',segundos=time.perf_counter()-inicio))
        if t10 % 10 == 0 or t10 == round(Tf * 10):
            for gpu in range(num_gpus):
                print(f'GPU {gpu} | estado {gpu+3} | completado {T:.1f} K | tiempo total {time.perf_counter()-inicio_total:.1f} s', flush=True)
finally:
    parar.set(); thread.join(timeout=5)
    (OUT/'tiempos.json').write_text(json.dumps(tiempos,indent=2)); (OUT/'memoria.json').write_text(json.dumps(monitor_datos,indent=2))
    print('Tiempo y memoria registrados en',OUT,flush=True)


In [ ]:
import pandas as pd
display(pd.DataFrame(tiempos))
print('Pico RAM (MiB):', max((x['rss_mib'] for x in monitor_datos),default=0))
gpu=[g for fila in monitor_datos for g in fila['gpu']]
if gpu: display(pd.DataFrame(gpu).groupby('indice')[['usada_mib','total_mib','utilizacion']].max())
print('Tamaño guardado (MiB):',sum(f.stat().st_size for f in OUT.rglob('*') if f.is_file())/2**20)